In [ ]:
from pathlib import Path
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator

# Trading Performance Analytics Dashboard
**Project 1 - Pandas + NumPy + Matplotlib.** Cleans a 40-row forex trading journal, measures performance and risk, and builds a dashboard.

Folder layout (all paths are relative, nothing is tied to one computer):
`data/raw/` original CSV (never edited) | `data/processed/` cleaned CSV | `figures/` PNG + PDF | `reports/` data-quality report, cleaning log, written report.

In [ ]:
# ---- Project configuration: relative paths only ----
from matplotlib.ticker import FuncFormatter

RAW_NAME = 'trading_journal_raw_project1.csv'
ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent):            # works from project root or from a notebooks/ subfolder
    if (candidate / 'data' / 'raw' / RAW_NAME).exists() or (candidate / RAW_NAME).exists():
        ROOT = candidate
        break
RAW_PATH = ROOT / 'data' / 'raw' / RAW_NAME
if not RAW_PATH.exists():
    RAW_PATH = ROOT / RAW_NAME                   # fallback: CSV next to the notebook
PROCESSED_DIR, FIG_DIR, REPORT_DIR = ROOT / 'data' / 'processed', ROOT / 'figures', ROOT / 'reports'
for folder in (PROCESSED_DIR, FIG_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Shared look for EVERY chart in the notebook (set once, before any plotting)
GAIN, LOSS = '#2a78d6', '#eb6834'
INK, INK2, GRID, TILE = '#0b0b0b', '#52514e', '#e6e5e1', '#f4f3ef'
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.edgecolor': '#9a9993',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID,
                     'grid.linewidth': 0.8, 'axes.axisbelow': True, 'text.color': INK, 'axes.labelcolor': INK2,
                     'xtick.color': INK2, 'ytick.color': INK2, 'legend.frameon': False, 'text.parse_math': False})
money = FuncFormatter(lambda v, _: f'-${abs(v):,.0f}' if v < 0 else f'${v:,.0f}')

def style_date_axis(ax, interval=3):
    # one consistent date-axis recipe: a tick every `interval` days, labelled month-day
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

def df_to_md(frame):
    # tiny markdown-table writer so we only need pandas/numpy/matplotlib
    cols = [str(c) for c in frame.columns]
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '|'.join(['---'] * len(cols)) + '|']
    for _, row in frame.iterrows():
        lines.append('| ' + ' | '.join(str(v).replace('|', '/') for v in row) + ' |')
    return '\n'.join(lines)

In [ ]:
dataframe = pd.read_csv(RAW_PATH)   # raw file is never modified
df = dataframe.copy()
print('Loaded', RAW_PATH.name, '->', df.shape[0], 'rows x', df.shape[1], 'columns')

## Stage 1 - Data inspection and validation

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
def uni_que(df, column_name):
    return df[column_name].unique()

In [ ]:
unique_direction = uni_que(df,'Direction')
unique_instrument = uni_que(df,'Instrument')
unique_trade_ID = uni_que(df,'Trade_ID')

print(unique_trade_ID)
print(unique_instrument)
print(unique_direction)

the trades are sequential but T039 is not fouund.Needs to verify from the actual trading log to see what happened

In [ ]:
matches_format = df['Trade_ID'].str.match(r'^T\d{3,}$', na=False)

In [ ]:
# ---- Stage 1: data-quality findings (run on the RAW data) ----
raw = dataframe
def ids_of(mask, frame=raw):
    return ', '.join(frame.loc[mask, 'Trade_ID'].astype(str).tolist()) or '-'
unique_rows = raw.drop_duplicates()
conflict_mask = unique_rows['Trade_ID'].duplicated(keep=False)

canon_instr = raw['Instrument'].str.strip().str.replace('/', '', regex=False).str.upper()
num = pd.to_numeric(raw['Trade_ID'].str.extract(r'(\d+)')[0], errors='coerce')
missing_ids = [f'T{n:03d}' for n in sorted(set(range(int(num.min()), int(num.max()) + 1)) - set(num.dropna().astype(int)))]
bad_date = pd.to_datetime(raw['Trade_Date'], errors='coerce').isna()
exact_dup = raw.duplicated(keep='first')
id_dup = raw['Trade_ID'].duplicated(keep=False)

num_cols = ['Entry_Price', 'Exit_Price', 'Position_Size_Lots', 'Realized_PnL_USD']
coerced = raw[num_cols].apply(pd.to_numeric, errors='coerce')
bad_numeric = coerced.isna() & raw[num_cols].notna()      # a value is present but is not a number
is_inf = np.isinf(coerced)
findings = [
 ('Missing Exit_Price',        int(raw['Exit_Price'].isna().sum()),           ids_of(raw['Exit_Price'].isna()),           'P&L and drawdown cannot be checked for the trade', 'Derive from P&L, flag as imputed'),
 ('Missing Position_Size_Lots', int(raw['Position_Size_Lots'].isna().sum()),  ids_of(raw['Position_Size_Lots'].isna()),  'P&L cannot be reconciled to a size',               'Derive from P&L, flag as imputed'),
 ('Missing Realized_PnL_USD',  int(raw['Realized_PnL_USD'].isna().sum()),     ids_of(raw['Realized_PnL_USD'].isna()),     'Would silently shrink net P&L if dropped',         'Derive from prices and size, flag as imputed'),
 ('Exact duplicate row',       int(exact_dup.sum()),                          ids_of(exact_dup),                          'Double-counts a trade (inflates P&L and trade count)', 'Remove the repeated copy only'),
 ('Same Trade_ID, different values', int(unique_rows['Trade_ID'].duplicated().sum()), ids_of(conflict_mask, unique_rows), 'Cannot tell which record is true', 'Investigate - none found'),
 ('Gap in Trade_ID sequence',  len(missing_ids),                              ', '.join(missing_ids) or '-',              'A real trade may be absent, so totals could be incomplete', 'Do not invent it; verify against broker log'),
 ('Invalid Trade_ID format',   int((~raw['Trade_ID'].str.match(r'^T\d{3,}$', na=False)).sum()), ids_of(~raw['Trade_ID'].str.match(r'^T\d{3,}$', na=False)), 'Breaks joins and sorting', 'None found'),
 ('Inconsistent Instrument text', int((raw['Instrument'] != canon_instr).sum()), ids_of(raw['Instrument'] != canon_instr), 'EURUSD would be split into 3 groups', 'Standardize to EURUSD, keep raw column'),
 ('Inconsistent Direction text', int((~raw['Direction'].isin(['BUY', 'SELL'])).sum()), ids_of(~raw['Direction'].isin(['BUY', 'SELL'])), 'BUY and Buy counted as different directions', 'Standardize to Buy/Sell, keep raw column'),
 ('Unparseable Trade_Date',    int(bad_date.sum()),                           ids_of(bad_date),                           'Breaks time-series analysis', 'None found'),
 ('Non-positive price',        int((raw[['Entry_Price', 'Exit_Price']] <= 0).any(axis=1).sum()), ids_of((raw[['Entry_Price', 'Exit_Price']] <= 0).any(axis=1)), 'Impossible market price', 'None found'),
 ('Non-positive position size', int((raw['Position_Size_Lots'] <= 0).sum()),  ids_of(raw['Position_Size_Lots'] <= 0),     'Impossible position size', 'None found'),
 ('Non-numeric entry in a numeric column', int(bad_numeric.any(axis=1).sum()), ids_of(bad_numeric.any(axis=1)), 'Text such as "n/a" would stop arithmetic or silently become NaN', 'None found'),
 ('Infinite value in a numeric column',   int(is_inf.any(axis=1).sum()),        ids_of(is_inf.any(axis=1)),        'Infinity would corrupt sums, drawdown and averages', 'None found'),
]
dq_report = pd.DataFrame(findings, columns=['Issue', 'Records', 'Trade_IDs', 'Potential impact', 'Planned action'])
dq_report

T007 (an EURUSD trade priced like GBPUSD) is not visible from the raw checks above; it is caught later by the price-range check in Stage 2.

In [ ]:
cleaned = df.copy()

In [ ]:
cleaned['Instrument'] = cleaned['Instrument'].str.strip().str.replace("/", "", regex=False).str.upper()
cleaned['Direction'] = cleaned['Direction'].astype('string').str.strip().str.title()


In [ ]:
cleaned['Trade_Date']=pd.to_datetime(cleaned['Trade_Date'], errors='coerce')

In [ ]:
cleaned.isna().sum()

In [ ]:
cleaned.loc[cleaned.isna().any(axis=1)]

Three missing values ie. T025(Exit_Price),T031(Position_size_lots),T036(Realized_PnL_USD). I need to investigate why they are missing.
Going forward I will used the formula to impute the following numbers and try to find the actual trading journal if available and investigate why they are missing

In [ ]:
cleaned[cleaned.duplicated(keep=False)]

One duplicate trade, same values for the two rows...... i would be dropping one for now to make calculations realistic.

In [ ]:
cleaned = cleaned.drop_duplicates()

In [ ]:
# Keep the raw labels next to the standardized ones (audit trail).  drop_duplicates() kept the original index.
cleaned['Instrument_Raw'] = df.loc[cleaned.index, 'Instrument']
cleaned['Direction_Raw'] = df.loc[cleaned.index, 'Direction']

# Conflicting trade IDs: same ID but different values would survive drop_duplicates().
conflicts = cleaned[cleaned['Trade_ID'].duplicated(keep=False)]
print('Conflicting Trade_ID records after removing exact duplicates:', len(conflicts))
assert cleaned['Trade_ID'].is_unique, 'Conflicting Trade_IDs need manual investigation'

I need to fill the missing values using this formula 'Profit = (exit_price - entry_price) * position lot size * standard lot(100000) * direction' 

In [ ]:
standard_lot = 100000
conditions = [
    cleaned['Direction'] == 'Buy',
    cleaned['Direction'] == 'Sell'
]
direction = np.select(conditions, [1, -1], default=np.nan)
assert not np.isnan(direction).any(), "Invalid or missing Direction values found"
cleaned['Imputed'] = ''

In [ ]:
missing_lot = cleaned['Position_Size_Lots'].isnull()

In [ ]:
cleaned.loc[missing_lot, 'Imputed'] = 'Lot_size_Imputed'

calculated_lots = (
    cleaned['Realized_PnL_USD'] /
    (direction * (cleaned['Exit_Price'] - cleaned['Entry_Price']) * standard_lot)
).round(2)

cleaned['Position_Size_Lots'] = (
    cleaned['Position_Size_Lots'].fillna(calculated_lots))

In [ ]:
missing_exit_price = cleaned['Exit_Price'].isnull()
cleaned.loc[missing_exit_price, 'Imputed'] = 'Exit_Price_Imputed'

calculated_exit_price = ((cleaned['Entry_Price']) + (cleaned['Realized_PnL_USD'])/(standard_lot * direction * cleaned['Position_Size_Lots'])).round(5)
cleaned['Exit_Price']= (cleaned['Exit_Price'].fillna(calculated_exit_price))

In [ ]:
missing_PnL = cleaned['Realized_PnL_USD'].isnull()
cleaned.loc[missing_PnL, 'Imputed'] = 'PnL_Imputed'

calculated_pnl = (
    cleaned['Position_Size_Lots'] * direction * (cleaned['Exit_Price'] - cleaned['Entry_Price']) * standard_lot
).round(2)
cleaned['Realized_PnL_USD'] = (
    cleaned['Realized_PnL_USD'].fillna(calculated_pnl))

In [ ]:
cleaned[cleaned['Imputed'] != ''][
    ['Trade_ID', 'Instrument', 'Direction', 'Entry_Price', 'Exit_Price',
     'Position_Size_Lots', 'Realized_PnL_USD', 'Imputed']]

In [ ]:
cleaned.isnull().sum()

I need to compare the realized Pnl to the actual pnl and the pips gained(it might help in future calcualtions)

In [ ]:
pip_size = np.where(cleaned["Instrument"].str.endswith('JPY'), 0.01, 0.0001)
is_buy = cleaned['Direction'] == 'Buy'
is_sell = cleaned['Direction'] == 'Sell'

In [ ]:
is_usd = np.select(
      [is_buy, is_sell],
      [
          (cleaned["Exit_Price"] - cleaned["Entry_Price"]) * cleaned['Position_Size_Lots'] * standard_lot,
          (cleaned["Entry_Price"] - cleaned["Exit_Price"]) * cleaned['Position_Size_Lots'] * standard_lot
      ],
      default=np.nan,
  )

In [ ]:
cleaned['Estimated_PnL'] = np.where(cleaned['Instrument'].str.endswith('JPY'), is_usd/cleaned['Exit_Price'],is_usd).round(2)

In [ ]:
cleaned["pips_gained_lost"] = np.select(
      [is_buy, is_sell],
      [
          (cleaned["Exit_Price"] - cleaned["Entry_Price"]) / pip_size,
          (cleaned["Entry_Price"] - cleaned["Exit_Price"]) / pip_size,
      ],
      default=np.nan,
  )

In [ ]:
cleaned.shape

In [ ]:
cleaned.isnull().sum()

In [ ]:
cleaned.duplicated().sum()

In [ ]:
cleaned.dtypes

In [ ]:
cleaned['Reconciliation_Diff'] = cleaned['Estimated_PnL'] - cleaned['Realized_PnL_USD']
TOLERANCE = 0.50
cleaned['Within_Tolerance'] = cleaned['Reconciliation_Diff'].abs() <= TOLERANCE
cleaned['Recon_Status'] = np.where(cleaned['Within_Tolerance'], 'Matched', 'Break')

In [ ]:
cleaned

In [ ]:
exception_report = cleaned[cleaned['Recon_Status'] == 'Break'].copy()

In [ ]:
print('\n=== EXCEPTION REPORT (ACTION REQUIRED) ===')
if exception_report.empty:
  print('No exceptions found. All trades within tolerance.')
else:
  print(exception_report.to_string(index=False))

### Internal consistency checks
Every check must pass. These test that the data is internally consistent (valid, finite, sensible); they do **not** verify records against the broker. For the 3 imputed rows the P&L reconciliation is circular (the missing field was computed from the other two), so those rows match by construction and are flagged rather than treated as confirmed.

In [ ]:
validation = {
    'Trade dates parsed':                  cleaned['Trade_Date'].notna().all(),
    'Dates inside journal window (Sep-Oct 2026)': cleaned['Trade_Date'].between('2026-09-01', '2026-10-31').all(),
    'Numeric columns have a numeric dtype': all(pd.api.types.is_numeric_dtype(cleaned[c]) for c in num_cols + ['Estimated_PnL', 'pips_gained_lost']),
    'All numeric values finite (no NaN, no +/-inf)': bool(np.isfinite(cleaned[num_cols + ['Estimated_PnL', 'pips_gained_lost']].to_numpy(float)).all()),
    'Prices finite and > 0':               bool((np.isfinite(cleaned[['Entry_Price', 'Exit_Price']]) & (cleaned[['Entry_Price', 'Exit_Price']] > 0)).all().all()),
    'Position sizes finite and > 0':       bool((np.isfinite(cleaned['Position_Size_Lots']) & (cleaned['Position_Size_Lots'] > 0)).all()),
    'Direction is Buy/Sell':               cleaned['Direction'].isin(['Buy', 'Sell']).all(),
    'Instrument in known set':             cleaned['Instrument'].isin(['EURUSD', 'GBPUSD', 'USDJPY']).all(),
    'P&L sign matches price move':         bool((np.sign(cleaned['Estimated_PnL']) == np.sign(cleaned['Realized_PnL_USD'])).all()),
    'Trade_ID unique':                     cleaned['Trade_ID'].is_unique,
}
for name, ok in validation.items():
    print(('PASS  ' if ok else 'FAIL  ') + name)
assert all(validation.values()), 'A validation check failed - investigate before analysis'

I want to check if all the trades has its correct symbol and not mixed up

In [ ]:
median_price = cleaned.groupby('Instrument')['Entry_Price'].transform('median')

suspect = (
    ((cleaned['Entry_Price'] / median_price - 1).abs() > 0.05) |
    ((cleaned['Exit_Price']  / median_price - 1).abs() > 0.05)
)

print(cleaned.loc[suspect, ['Trade_ID','Instrument', 'Entry_Price', 'Exit_Price']])

After running the code,  trade T007 has and entry and exit price similar to GBPUSD it needs to be looked into it. I will confirm that from the brokers log to see if it was a typo

In [ ]:
# ---- Quality flags: a record that cannot be validated is flagged, never silently trusted ----
cleaned['Quality_Status'] = 'Passed_Checks'
cleaned['Quality_Note'] = ''
cleaned.loc[cleaned['Imputed'] != '', 'Quality_Status'] = 'Imputed_Unverified'
cleaned.loc[cleaned['Imputed'] != '', 'Quality_Note'] = cleaned['Imputed'].str.replace('_', ' ') + ' (derived from the other fields, not from a broker record)'
cleaned.loc[suspect, 'Quality_Status'] = 'Needs_Review'
cleaned.loc[suspect, 'Quality_Note'] = 'Entry/exit prices far from this instrument\'s median (GBPUSD-like) - possible wrong symbol'
cleaned['Quality_Status'].value_counts()

In [ ]:
# ---- Cleaning log + exports ----
n_instr = int((cleaned['Instrument'] != cleaned['Instrument_Raw']).sum())
n_dir = int((cleaned['Direction'] != cleaned['Direction_Raw']).sum())
def ids(mask):
    return ', '.join(cleaned.loc[mask, 'Trade_ID']) or '-'
cleaning_log = pd.DataFrame([
 ('1', 'Standardize Instrument (trim, remove "/", upper-case)', n_instr, ids(cleaned['Instrument'] != cleaned['Instrument_Raw']), 'Same instrument written 3 ways would split EURUSD results. Raw text kept in Instrument_Raw.'),
 ('2', 'Standardize Direction to Buy/Sell', n_dir, ids(cleaned['Direction'] != cleaned['Direction_Raw']), 'Case differences are formatting only. Raw text kept in Direction_Raw.'),
 ('3', 'Parse Trade_Date to datetime', len(cleaned), 'all', 'Needed for sorting and time series; no failures.'),
 ('4', 'Remove exact duplicate row', int(exact_dup.sum()), ids_of(exact_dup), 'Every field identical, so it is a repeated copy. Only exact copies were removed.'),
 ('5', 'Impute missing Exit_Price (T025)', 1, ids(cleaned['Imputed'] == 'Exit_Price_Imputed'), 'Entry + P&L / (lots x 100000 x direction). Flagged Imputed_Unverified; confirm with broker log.'),
 ('6', 'Impute missing Position_Size_Lots (T031)', 1, ids(cleaned['Imputed'] == 'Lot_size_Imputed'), 'P&L / (price move x 100000 x direction). Flagged Imputed_Unverified.'),
 ('7', 'Impute missing Realized_PnL_USD (T036)', 1, ids(cleaned['Imputed'] == 'PnL_Imputed'), 'Lots x price move x 100000 x direction. Flagged Imputed_Unverified.'),
 ('8', 'Flag suspect instrument/price (not changed)', int(suspect.sum()), ids(suspect), 'Prices look like GBPUSD. Kept as recorded and flagged Needs_Review; sensitivity shown in Stage 4.'),
 ('9', 'Record missing Trade_ID gap (no row created)', len(missing_ids), ', '.join(missing_ids), 'Possibly an unrecorded trade. Not invented; verify with the broker.'),
], columns=['Step', 'Transformation / exclusion', 'Records affected', 'Trade_IDs', 'Justification'])

export_cols = ['Trade_ID', 'Trade_Date', 'Instrument', 'Direction', 'Entry_Price', 'Exit_Price', 'Position_Size_Lots',
               'Realized_PnL_USD', 'Instrument_Raw', 'Direction_Raw', 'Imputed', 'Estimated_PnL', 'pips_gained_lost',
               'Recon_Status', 'Quality_Status', 'Quality_Note']
cleaned[export_cols].to_csv(PROCESSED_DIR / 'trading_journal_cleaned.csv', index=False)
cleaning_log.to_csv(REPORT_DIR / 'cleaning_log.csv', index=False)
dq_report.to_csv(REPORT_DIR / 'data_quality_report.csv', index=False)
(REPORT_DIR / 'data_quality_and_cleaning_log.md').write_text(
    '# Data-quality report\n\n' + df_to_md(dq_report) + '\n\n# Cleaning log\n\n' + df_to_md(cleaning_log) + '\n')

check = pd.read_csv(PROCESSED_DIR / 'trading_journal_cleaned.csv')
assert len(check) == len(cleaned) == 39
cleaning_log

In [ ]:
sorted_values = cleaned.sort_values(by=['Trade_Date',"Trade_ID"] ,ascending=[True,True])

In [ ]:
trades = sorted_values[['Trade_ID','Direction','Realized_PnL_USD']].to_numpy()
print(trades)


In [ ]:
trades.shape

## Stage 3 - Exploratory data analysis

In [ ]:
eda = sorted_values
print('Records:', len(eda), '| unique Trade_IDs:', eda['Trade_ID'].nunique())
print('\nTrades per instrument:'); print(eda['Instrument'].value_counts().to_string())
print('\nTrades per direction:');  print(eda['Direction'].value_counts().to_string())
first, last = eda['Trade_Date'].min(), eda['Trade_Date'].max()
print(f'\nDate coverage: {first:%Y-%m-%d} to {last:%Y-%m-%d} ({(last - first).days + 1} calendar days, {eda["Trade_Date"].nunique()} trading days)')
print('Trades per trading day: min', eda.groupby('Trade_Date').size().min(), '| max', eda.groupby('Trade_Date').size().max())

In [ ]:
pnl_eda = eda['Realized_PnL_USD']
print('P&L distribution (USD per trade)')
print(pnl_eda.describe().round(2).to_string())
print('skew:', round(pnl_eda.skew(), 2), '| median vs mean:', round(pnl_eda.median(), 2), 'vs', round(pnl_eda.mean(), 2))
print('\nPosition size (lots) value counts:')
print(eda['Position_Size_Lots'].value_counts().sort_index().to_string())

In [ ]:
def summarize_by(col):
    rows = {}
    for key, g in eda.groupby(col):
        p = g['Realized_PnL_USD'].to_numpy()
        rows[key] = {'trades': len(p), 'wins': int((p > 0).sum()), 'win_rate_pct': 100 * (p > 0).mean(),
                     'net_pnl': p.sum(), 'avg_pnl': p.mean(), 'best': p.max(), 'worst': p.min(),
                     'avg_pips': g['pips_gained_lost'].mean()}
    return pd.DataFrame(rows).T.astype(float).round(2)

by_instrument = summarize_by('Instrument')
by_direction = summarize_by('Direction')
by_size = summarize_by('Position_Size_Lots')
print('By instrument'); print(by_instrument.to_string())
print('\nBy direction');  print(by_direction.to_string())
print('\nBy position size (lots)'); print(by_size.to_string())

In [ ]:
cols = ['Trade_ID', 'Trade_Date', 'Instrument', 'Direction', 'Position_Size_Lots', 'Realized_PnL_USD', 'Quality_Status']
print('Largest winners');  print(eda.nlargest(5, 'Realized_PnL_USD')[cols].to_string(index=False))
print('\nLargest losers'); print(eda.nsmallest(5, 'Realized_PnL_USD')[cols].to_string(index=False))

In [ ]:
# Outliers are FLAGGED for investigation, never deleted automatically.
p = eda['Realized_PnL_USD'].to_numpy()
q1, q3 = np.percentile(p, [25, 75]); iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
z = (p - p.mean()) / p.std(ddof=1)
flag = (p < lo) | (p > hi) | (np.abs(z) > 2)
print(f'IQR fences: [{lo:.2f}, {hi:.2f}]  |  z-score cut-off: 2')
print(eda.assign(z_score=z.round(2))[flag][['Trade_ID', 'Instrument', 'Position_Size_Lots', 'Realized_PnL_USD', 'z_score', 'Quality_Status']].to_string(index=False))

• Calculate total trades, wins, losses, breakeven trades, and net realized P&L.
• Calculate win rate, loss rate, average win, average loss, and average P&L per trade.
• Calculate gross profit, gross loss, profit factor, and trading expectancy.
• Identify the best and worst trades by realized P&L.
• Explain assumptions about missing P&L, fees, breakeven trades, and excluded records.

In [ ]:
pnl = trades[:,2].astype(float)
total_trades = np.size(pnl)
total_wins = np.sum(pnl > 0)
total_loss = np.sum(pnl<0)
breakeven = np.sum(pnl == 0)
net_profit = np.sum(pnl) 
print('Total Wins:', total_wins)
print('Total Trades:', total_trades)
print('Total Loss:', total_loss)
print('Breakeven Trades:', breakeven)
print('Net Profit:', net_profit.round(2))

In [ ]:
win_rate = ((total_wins / (total_trades))*100)
loss_rate = ((total_loss/total_trades)*100)
average_win = (np.mean(pnl[pnl > 0]))
average_loss = (np.mean(pnl[pnl<0]))
average_pl = (np.mean(pnl))
print("average P&L per trade:", average_pl.round(2))
print("Average Win:", average_win.round(2))
print("Average Loss", average_loss.round(2))
print('Win_Rate', win_rate.round(2))
print('Loss Rate', loss_rate.round(2))

In [ ]:
gross_profit = pnl[pnl > 0].sum()
gross_loss = pnl[pnl < 0].sum()
expectancy = ((win_rate * average_win) - (loss_rate * abs(average_loss))) / 100
print('Gross Loss', round(gross_loss,2))
print('Gross profit', round(gross_profit,2))
print("Profit factor:", round(gross_profit / abs(gross_loss), 2))
print('Expectancy Per Trade:', round(expectancy,2))

In [ ]:
best_trade = np.max(pnl)
worst_trade = np.min(pnl)
print("Worst Trade:", worst_trade)
print ('Best Trade:', best_trade)

In [ ]:
rank_cols = ['Trade_ID', 'Trade_Date', 'Instrument', 'Direction', 'Position_Size_Lots', 'Realized_PnL_USD']
print('Best 3 trades');  print(sorted_values.nlargest(3, 'Realized_PnL_USD')[rank_cols].to_string(index=False))
print('\nWorst 3 trades'); print(sorted_values.nsmallest(3, 'Realized_PnL_USD')[rank_cols].to_string(index=False))

# Internal consistency checks on the metrics above
assert total_wins + total_loss + breakeven == total_trades
assert np.isclose(net_profit, gross_profit + gross_loss)
assert np.isclose(expectancy, average_pl)       # expectancy == mean P&L per trade
assert np.isclose(net_profit, sorted_values['Realized_PnL_USD'].sum())

**Assumptions behind the Stage 4 numbers**
- **Data basis:** 39 retained trade records after removing the exact T011 duplicate. Three records have a reconstructed field (T025, T031, T036), and T007 has a suspicious instrument label. The other 35 records passed internal consistency checks, but **none have been independently verified against broker records**. T039 is absent from the ID sequence; this does not prove a trade is missing.
- **Reconstructed values:** Missing price, size and P&L fields were computed from other fields, so reversing the formula confirms internal arithmetic only, **not** an actual broker execution. Sensitivity tests compare results with/without flagged records.
- **Trading costs:** The journal has no separately recorded commissions, swaps or spread charges and does not indicate whether these are already reflected in recorded P&L. Accordingly all metrics describe **recorded P&L**, without claiming to be gross or net of all trading costs.
- **Conversion assumptions:** For USDJPY, the quote-price P&L is in JPY and is approximately converted to USD using the exit exchange rate. The broker may use a different settlement rate and rounding; reconciliation tolerance does not independently validate executions.
- **Execution sequence:** Same-day trades are sorted by Trade_ID because execution timestamps are unavailable. Trade-level maximum drawdown is conditional on that ordering assumption.
- **Definitions:** Profit factor = gross profit / absolute gross loss. Sample expectancy = mean P&L per trade. This small journal does not establish a long-term trading edge.


In [ ]:
# Sensitivity: do the conclusions depend on the doubtful records?
def max_dd(p):
    eq = np.cumsum(p)
    return float((np.maximum(0, np.maximum.accumulate(eq)) - eq).max())

def perf_metrics(p):
    p = np.asarray(p, float); w, l = p[p > 0], p[p < 0]
    return {'trades': p.size, 'net_pnl': p.sum(), 'win_rate_pct': 100 * w.size / p.size,
            'avg_win': w.mean(), 'avg_loss': l.mean(), 'profit_factor': w.sum() / abs(l.sum()),
            'expectancy': p.mean(), 'max_drawdown': max_dd(p)}

sv = sorted_values
scenarios = {
    'All 39 trades (base case)':        np.ones(len(sv), bool),
    'Excluding 3 imputed trades':       (sv['Quality_Status'] != 'Imputed_Unverified').to_numpy(),
    'Excluding T007 (suspect symbol)':  (sv['Quality_Status'] != 'Needs_Review').to_numpy(),
    'Excluding all 4 flagged trades':   (sv['Quality_Status'] == 'Passed_Checks').to_numpy(),
}
sensitivity = pd.DataFrame({k: perf_metrics(sv.loc[m, 'Realized_PnL_USD']) for k, m in scenarios.items()}).T.round(2)
assert np.isclose(sensitivity.iloc[0]['net_pnl'], net_profit)
sensitivity

## Stage 5 - Cumulative P&L and drawdown analysis
**Ordering:** trades are sorted by `Trade_Date`, then `Trade_ID`. The journal has no timestamps, so Trade_ID order is assumed to be execution order within a day (6 trades on 2026-09-21, 5 on 2026-09-28). **Baseline:** the curve starts at $0 of cumulative realized P&L (account size is unknown). Variables named `equity` below mean this cumulative realized P&L, not account equity.

In [ ]:

cumulative = np.cumsum(pnl)
print(cumulative)

In [ ]:
# Cumulative realized P&L, one point per trade, against trade date (same-day trades share a date)
base = 0
fig, ax = plt.subplots(figsize=(10, 5))
ax.axhline(base, linestyle='--', color=INK2, lw=1)
ax.plot(sorted_values['Trade_Date'], cumulative, color=GAIN, marker='o', ms=4, lw=1.5)
ax.set_xlabel('Trade date (2026, month-day); one point per trade')
ax.set_ylabel('Cumulative realized P&L (USD)')
ax.yaxis.set_major_formatter(money)
style_date_axis(ax)
ax.set_title('Cumulative realized P&L by trade date', loc='left')
plt.tight_layout(); plt.show()

We don't know the initial captial we are doing the running peak without the initial capital

In [ ]:
# NOTE: 'equity' here means cumulative realized P&L from a $0 baseline - it is NOT account equity (no account balance exists)
equity = base + cumulative
print(equity)

In [ ]:
# running peak of cumulative realized P&L, floored at the $0 starting baseline

peak_account = np.maximum(0,np.maximum.accumulate(equity))
print(peak_account)

In [ ]:
drawdown = equity - peak_account
print(drawdown)

In [ ]:
#max drawdown

max_drawdown = np.min(drawdown)
print(abs(max_drawdown))

In [ ]:

trade_number = np.argmin(drawdown)
print(trade_number)

In [ ]:
trough_i = int(np.argmin(drawdown))
peak_val = float(peak_account[trough_i])
peak_i = int(np.argmax(equity[:trough_i + 1])) if peak_val > 0 else None
sv = sorted_values.reset_index(drop=True)
assert np.isclose(abs(drawdown.min()), (peak_account - equity).max())      # two ways to get max drawdown agree

print(f'Maximum absolute drawdown: ${abs(drawdown.min()):,.2f}')
print(f'Trough: trade {sv.loc[trough_i, "Trade_ID"]} on {sv.loc[trough_i, "Trade_Date"]:%Y-%m-%d} (cumulative P&L ${equity[trough_i]:,.2f})')
if peak_i is not None:
    print(f'Peak before it: trade {sv.loc[peak_i, "Trade_ID"]} on {sv.loc[peak_i, "Trade_Date"]:%Y-%m-%d} (cumulative P&L ${equity[peak_i]:,.2f}), {trough_i - peak_i} trades earlier')
print('Recovered to that peak by the last trade?', bool((equity[trough_i:] >= peak_val).any()))
print(f'Drawdown at the last trade: ${drawdown[-1]:,.2f} | trades spent below the running peak: {(drawdown < 0).sum()} of {len(drawdown)}')

**Absolute vs percentage drawdown.** The figure above is in dollars: the fall from the highest cumulative P&L to the lowest point after it. A percentage drawdown needs an account balance (drawdown / peak *account value*), and this journal has no starting capital or deposits. Dividing by the tiny $60.76 peak of *profit* would give a meaningless -700%, so **no percentage is reported**. If the starting balance becomes known, add it to the baseline and the same code gives a real percentage.

## Stage 6 - Time-series analysis

In [ ]:
daily_pnl = sorted_values.groupby('Trade_Date')['Realized_PnL_USD'].agg(total_pnl='sum', trade_count='count').reset_index()
daily_pnl

In [ ]:
monthly_pnl = (
    sorted_values.groupby(sorted_values['Trade_Date'].dt.to_period('M'))['Realized_PnL_USD']
    .agg(total_pnl='sum', trade_count='count')
    .reset_index())

monthly_pnl

In [ ]:
daily_pnl['rolling_5session_avg'] = (
    daily_pnl['total_pnl']
    .rolling(window=5)
    .mean())

In [ ]:
sorted_values['rolling_5trade_avg'] =(sorted_values['Realized_PnL_USD'].rolling(window=5).mean())

In [ ]:
sorted_values['rolling_5trade_avg']

In [ ]:
daily_pnl['rolling_5session_avg'].reset_index()

In [ ]:
daily_pnl

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(daily_pnl['Trade_Date'], daily_pnl['total_pnl'], width=0.7, color=np.where(daily_pnl['total_pnl'] >= 0, GAIN, LOSS))
ax.plot(daily_pnl['Trade_Date'], daily_pnl['rolling_5session_avg'], color=INK2, lw=2, marker='s', ms=4)
ax.axhline(0, color=INK2, lw=1)
ax.set_title('Daily realized P&L vs 5-session rolling average', loc='left')
ax.set_xlabel('Trading day (2026, month-day)'); ax.set_ylabel('Realized P&L (USD)')
ax.yaxis.set_major_formatter(money); style_date_axis(ax)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=GAIN), plt.Rectangle((0, 0), 1, 1, color=LOSS), plt.Line2D([], [], color=INK2, lw=2)],
          labels=['Gain day', 'Loss day', '5-session rolling average'], loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
# longest winning and losing streaks
def longest_run(mask):
    best = current = 0
    for x in mask:
        current = current + 1 if x else 0
        best = max(best, current)
    return best

longest_win_streak = longest_run(sorted_values['Realized_PnL_USD'] > 0)
longest_loss_streak = longest_run(sorted_values['Realized_PnL_USD'] < 0)

print("Longest winning streak:", longest_win_streak)
print("Longest losing streak:", longest_loss_streak)

In [ ]:
def run_spans(mask):
    # returns (start_index, length) of every consecutive run of True
    spans, start = [], None
    for i, v in enumerate(list(mask) + [False]):
        if v and start is None: start = i
        if not v and start is not None: spans.append((start, i - start)); start = None
    return spans

sv = sorted_values.reset_index(drop=True)
for label, mask in (('winning', sv['Realized_PnL_USD'] > 0), ('losing', sv['Realized_PnL_USD'] < 0)):
    s, n = max(run_spans(mask), key=lambda t: t[1])
    seg = sv.loc[s:s + n - 1]
    print(f'Longest {label} streak: {n} trades, {seg["Trade_ID"].iloc[0]} to {seg["Trade_ID"].iloc[-1]} '
          f'({seg["Trade_Date"].iloc[0]:%Y-%m-%d} to {seg["Trade_Date"].iloc[-1]:%Y-%m-%d}), net ${seg["Realized_PnL_USD"].sum():,.2f}')

worst_day = daily_pnl.loc[daily_pnl['total_pnl'].idxmin()]
best_day = daily_pnl.loc[daily_pnl['total_pnl'].idxmax()]
print(f'\nWorst day: {worst_day["Trade_Date"]:%Y-%m-%d} ${worst_day["total_pnl"]:,.2f} ({int(worst_day["trade_count"])} trades)')
print(f'Best day:  {best_day["Trade_Date"]:%Y-%m-%d} ${best_day["total_pnl"]:,.2f} ({int(best_day["trade_count"])} trades)')
print('Losing days:', int((daily_pnl['total_pnl'] < 0).sum()), 'of', len(daily_pnl))

# Change in performance: before vs after the drawdown trough
before_p = sv.loc[:trough_i, 'Realized_PnL_USD']; after_p = sv.loc[trough_i + 1:, 'Realized_PnL_USD']
print(f'\nTrades up to and including the trough ({sv.loc[trough_i, "Trade_ID"]}): {len(before_p)} trades, net ${before_p.sum():,.2f}, win rate {100*(before_p>0).mean():.0f}%')
print(f'Trades after the trough: {len(after_p)} trades, net ${after_p.sum():,.2f}, win rate {100*(after_p>0).mean():.0f}%')
# Rolling window check: lowest 5-session average
low = daily_pnl.loc[daily_pnl['rolling_5session_avg'].idxmin()]
print(f'Weakest 5-session average: ${low["rolling_5session_avg"]:,.2f}/day, ending {low["Trade_Date"]:%Y-%m-%d}')

**Limits of the time views.**
- *Daily* P&L groups trades by calendar date; days hold 1-6 trades, so a big day may just mean more trades.
- *Monthly* is almost useless here: September holds 38 trades and October a single one (T040), so comparing months compares 38 trades with 1.
- The **5-session rolling average** (about one trading week) smooths noise but needs 5 days before it starts, so the first four days are blank, and with only 16 trading days it has just 12 points. A longer window would leave almost nothing.
- Two different windows exist: `rolling_5trade_avg` (trade table) averages 5 **trades**; `rolling_5session_avg` (daily table) averages 5 **trading days**.
- Streaks and "turning points" come from 39 trades, so they can easily be luck.

In [ ]:
sorted_values['drawdown'] = drawdown

In [ ]:
sorted_values['drawdown']

In [ ]:
daily_dd = sorted_values.groupby('Trade_Date')['drawdown'].agg(dailydd='min', trade_count='count').reset_index()
daily_dd

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(daily_dd['Trade_Date'], daily_dd['dailydd'], 0, color=LOSS, alpha=0.3, lw=0)
ax.plot(daily_dd['Trade_Date'], daily_dd['dailydd'], color=LOSS, lw=2, marker='o', ms=4)
ax.axhline(0, color=INK2, lw=1, ls='--')
ax.set_title('Drawdown below the running peak (worst point of each trading day)', loc='left')
ax.set_xlabel('Trading day (2026, month-day)'); ax.set_ylabel('Drawdown (USD)')
ax.yaxis.set_major_formatter(money); style_date_axis(ax)
plt.tight_layout(); plt.show()

In [ ]:
sorted_values.groupby('Trade_Date')['Realized_PnL_USD'].count()

In [ ]:
m = sorted_values.groupby('Trade_Date')['Realized_PnL_USD'].sum().reset_index()

In [ ]:
m

In [ ]:
np.cumsum(m['Realized_PnL_USD'].to_numpy())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.axhline(0, color=INK2, lw=1, ls='--')
ax.plot(m['Trade_Date'], np.cumsum(m['Realized_PnL_USD'].to_numpy()), color=GAIN, lw=2, marker='o', ms=4)
ax.set_title('Cumulative realized P&L by trading day (end-of-day values)', loc='left')
ax.set_xlabel('Trading day (2026, month-day)'); ax.set_ylabel('Cumulative realized P&L (USD)')
ax.yaxis.set_major_formatter(money); style_date_axis(ax)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(daily_pnl['Trade_Date'], daily_pnl['total_pnl'], width=0.7, color=np.where(daily_pnl['total_pnl'] >= 0, GAIN, LOSS))
ax.axhline(0, color=INK2, lw=1)
ax.set_title('Daily realized P&L', loc='left')
ax.set_xlabel('Trading day (2026, month-day)'); ax.set_ylabel('Realized P&L (USD)')
ax.yaxis.set_major_formatter(money); style_date_axis(ax)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = monthly_pnl['Trade_Date'].astype(str)
vals = monthly_pnl['total_pnl'].to_numpy()
bars = ax.bar(labels, vals, width=0.5, color=np.where(vals >= 0, GAIN, LOSS))
ax.axhline(0, color=INK2, lw=1)
for bar, v, n in zip(bars, vals, monthly_pnl['trade_count']):
    ax.text(bar.get_x() + bar.get_width() / 2, v + (8 if v >= 0 else -8), f'{money(v, None)}\n{int(n)} trade' + ('' if int(n) == 1 else 's'),
            ha='center', va='bottom' if v >= 0 else 'top', fontsize=9)
ax.set_ylim(vals.min() * 1.4, max(vals.max(), 0) * 3)
ax.set_title('Monthly realized P&L (very unequal trade counts)', loc='left')
ax.set_xlabel('Month'); ax.set_ylabel('Realized P&L (USD)'); ax.yaxis.set_major_formatter(money)
ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

In [ ]:
sorted_values

## Stage 7 - Matplotlib visualizations
Colors: **blue = gain / cumulative P&L line, orange = loss**, grey for reference lines (a colour-blind-safe pair, always paired with a sign, label or position so colour is never the only cue).

In [ ]:
sv = sorted_values.reset_index(drop=True)
n_tr = len(sv)
x0 = np.arange(0, n_tr + 1)                      # 0 = starting baseline, then trade 1..n
eq0 = np.r_[0.0, equity]
dd0 = np.r_[0.0, drawdown]
trough_id = sv.loc[trough_i, 'Trade_ID']

def plot_equity(ax):
    ax.axhline(0, color=INK2, lw=1, ls='--')
    ax.plot(x0, eq0, color=GAIN, lw=2)
    ax.plot(trough_i + 1, equity[trough_i], 'o', color=LOSS, ms=8, mec='white', mew=2)
    ax.set_ylim(equity.min() * 1.3, max(equity.max(), 0) * 1.6)
    ax.annotate(f'Lowest point: {money(equity[trough_i], None)} ({trough_id})', (trough_i + 1, equity[trough_i]),
                xytext=(-10, -16), textcoords='offset points', ha='right', va='top', color=INK, fontsize=9)
    ax.annotate(f'End: {money(eq0[-1], None)}', (x0[-1], eq0[-1]), xytext=(0, 10), textcoords='offset points', ha='right', color=INK, fontsize=9)
    ax.set_title('Cumulative realized P&L (starts at $0)', loc='left')
    ax.set_xlabel('Trade number (chronological)'); ax.set_ylabel('Cumulative P&L (USD)')
    ax.yaxis.set_major_formatter(money); ax.set_xlim(0, n_tr)

def plot_underwater(ax):
    ax.fill_between(x0, dd0, 0, color=LOSS, alpha=0.35, lw=0)
    ax.plot(x0, dd0, color=LOSS, lw=2)
    ax.set_ylim(drawdown.min() * 1.3, 20)            # full range shown, nothing clipped
    ax.annotate(f'Max drawdown {money(drawdown.min(), None)}\n({trough_id}, {sv.loc[trough_i, "Trade_Date"]:%b %d})', (trough_i + 1, drawdown.min()),
                xytext=(0.62, 0.07), textcoords='axes fraction', ha='center', va='bottom', color=INK, fontsize=9,
                arrowprops=dict(arrowstyle='-', color=INK2, lw=1))
    ax.set_title('Underwater chart: distance below the running peak', loc='left')
    ax.set_xlabel('Trade number (chronological)'); ax.set_ylabel('Drawdown (USD)')
    ax.yaxis.set_major_formatter(money); ax.set_xlim(0, n_tr)

def plot_daily(ax):
    pos = np.arange(len(daily_pnl))
    cols = np.where(daily_pnl['total_pnl'] >= 0, GAIN, LOSS)
    ax.bar(pos, daily_pnl['total_pnl'], color=cols, width=0.7)
    ax.plot(pos, daily_pnl['rolling_5session_avg'], color=INK2, lw=2)
    ax.axhline(0, color=INK2, lw=1)
    ax.set_xticks(pos); ax.set_xticklabels(daily_pnl['Trade_Date'].dt.strftime('%m-%d'), rotation=60, ha='right', fontsize=8)
    ax.set_title('Daily realized P&L and 5-session average', loc='left')
    ax.set_xlabel('Trading day (2026, month-day)'); ax.set_ylabel('P&L (USD)'); ax.yaxis.set_major_formatter(money)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=GAIN), plt.Rectangle((0, 0), 1, 1, color=LOSS), plt.Line2D([], [], color=INK2, lw=2)],
              labels=['Gain day', 'Loss day', '5-session average'], loc='lower right', fontsize=8)

def plot_distribution(ax):
    edges = np.arange(np.floor(pnl.min() / 15) * 15, pnl.max() + 15, 15)
    counts, edges = np.histogram(pnl, bins=edges)
    centers = (edges[:-1] + edges[1:]) / 2
    ax.bar(centers, counts, width=13, color=np.where(centers >= 0, GAIN, LOSS))
    ax.axvline(0, color=INK2, lw=1)
    ax.axvline(pnl.mean(), color=INK, lw=1.5, ls='--')
    ax.text(pnl.mean(), counts.max() * 1.02, f' mean {money(pnl.mean(), None)}', ha='right', va='bottom', fontsize=9, color=INK)
    ax.set_title(f'Trade P&L distribution: {total_wins} wins vs {total_loss} losses', loc='left')
    ax.set_xlabel('Realized P&L per trade (USD, $15 bins)'); ax.set_ylabel('Number of trades')
    ax.xaxis.set_major_formatter(money); ax.set_ylim(0, counts.max() * 1.2)

def plot_group(ax, summary, title, xlabel):
    labels = list(summary.index.astype(str)); vals = summary['net_pnl'].to_numpy()
    bars = ax.bar(labels, vals, color=np.where(vals >= 0, GAIN, LOSS), width=0.55)
    ax.axhline(0, color=INK2, lw=1)
    span = np.abs(vals).max()
    for b, v, nt, wr in zip(bars, vals, summary['trades'], summary['win_rate_pct']):
        ax.text(b.get_x() + b.get_width() / 2, v + (0.04 * span if v >= 0 else -0.04 * span), f'{money(v, None)}\n{int(nt)} trades, {wr:.0f}% win',
                ha='center', va='bottom' if v >= 0 else 'top', fontsize=8.5, color=INK)
    ax.set_ylim(min(vals.min(), 0) - 0.4 * span, max(vals.max(), 0) + 0.4 * span)
    ax.set_title(title, loc='left'); ax.set_xlabel(xlabel); ax.set_ylabel('Net P&L (USD)'); ax.yaxis.set_major_formatter(money)
    ax.grid(axis='x', visible=False)

def plot_win_loss(ax):
    bars = ax.bar(['Wins', 'Losses'], [total_wins, total_loss], color=[GAIN, LOSS], width=0.5)
    for b, v, avg in zip(bars, [total_wins, total_loss], [average_win, average_loss]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.4, f'{v} trades\navg {money(avg, None)}', ha='center', va='bottom', fontsize=9, color=INK)
    ax.set_ylim(0, max(total_wins, total_loss) * 1.3)
    ax.set_title('Winning vs losing trades', loc='left'); ax.set_ylabel('Number of trades'); ax.grid(axis='x', visible=False)

fig, ax = plt.subplots(figsize=(9, 4.5)); plot_equity(ax); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(9, 4.5)); plot_underwater(ax); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5)); plot_daily(ax); plt.tight_layout(); plt.show()
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'width_ratios': [1.6, 1]})
plot_distribution(axs[0]); plot_win_loss(axs[1]); plt.tight_layout(); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
plot_group(axs[0], by_instrument, 'Net P&L by instrument', 'Instrument')
plot_group(axs[1], by_direction, 'Net P&L by direction', 'Direction')
plt.tight_layout(); plt.show()

**Why each chart**
- **Cumulative P&L line (trade number on x):** shows the path of results. Trade number rather than date because several trades share a date. Zero baseline is dashed so being below water is obvious.
- **Underwater area:** drawdown is easier to judge as a depth below zero than as the gap between two lines. The y-axis runs to the full depth, not truncated.
- **Daily bars + rolling line:** bars suit discrete daily totals (a line would imply continuity between days). Blue/orange follows the sign. Daily results are not smoothed away; the grey average is added on top.
- **Histogram + win/loss bars:** the histogram shows the *size* of results, which win rate alone hides; the count bars show frequency with average size labelled.
- **Instrument / direction bars:** compare totals across a handful of categories; each bar is labelled with its trade count because small groups can mislead.

## Stage 8 - Integrated performance dashboard

In [ ]:
def kpi_tile(ax, i, label, value, sub):
    w, gap = 0.155, 0.014; x = i * (w + gap)
    ax.add_patch(plt.Rectangle((x, 0.04), w, 0.92, transform=ax.transAxes, fc=TILE, ec='none'))
    ax.text(x + w / 2, 0.76, label, ha='center', va='center', fontsize=9.5, color=INK2, transform=ax.transAxes)
    ax.text(x + w / 2, 0.45, value, ha='center', va='center', fontsize=20, fontweight='bold', color=INK, transform=ax.transAxes)
    ax.text(x + w / 2, 0.17, sub, ha='center', va='center', fontsize=8.5, color=INK2, transform=ax.transAxes)

money_precise = lambda v: f'-${abs(v):,.2f}' if v < 0 else f'${v:,.2f}'
payoff = abs(average_win / average_loss)
fig = plt.figure(figsize=(17, 11.5), facecolor='white')
gs = fig.add_gridspec(2, 3, hspace=0.55, wspace=0.28, left=0.05, right=0.98, top=0.75, bottom=0.095)

axk = fig.add_axes([0.05, 0.795, 0.93, 0.095]); axk.axis('off')
kpi_tile(axk, 0, 'Net recorded P&L', money_precise(net_profit), f'{total_trades} trades')
kpi_tile(axk, 1, 'Max drawdown', money_precise(-abs(max_drawdown)), f'{trough_id}, {sv.loc[trough_i, "Trade_Date"]:%b %d}')
kpi_tile(axk, 2, 'Win rate', f'{win_rate:.1f}%', f'{total_wins} wins / {total_loss} losses')
kpi_tile(axk, 3, 'Profit factor', f'{gross_profit / abs(gross_loss):.2f}', '<1.0 = losing money')
kpi_tile(axk, 4, 'Expectancy / trade', money_precise(expectancy), f'avg win +{average_win:.0f} / avg loss {average_loss:.0f} (USD)')
kpi_tile(axk, 5, 'Avg win / avg loss', f'{payoff:.2f}', 'wins are smaller than losses')

plot_equity(fig.add_subplot(gs[0, 0]))
plot_underwater(fig.add_subplot(gs[0, 1]))
plot_daily(fig.add_subplot(gs[0, 2]))
plot_distribution(fig.add_subplot(gs[1, 0]))
plot_group(fig.add_subplot(gs[1, 1]), by_instrument, 'Net P&L by instrument', 'Instrument')
plot_group(fig.add_subplot(gs[1, 2]), by_direction, 'Net P&L by direction', 'Direction')

fig.suptitle('Trading Performance Analytics Dashboard', x=0.05, y=0.99, ha='left', fontsize=20, fontweight='bold')
flag_free = sensitivity.iloc[3]; n_clean = int(flag_free['trades'])
fig.text(0.05, 0.94, f'{total_trades} retained records ({n_clean} passed internal checks, {total_trades - n_clean} flagged: T025, T031, T036 derived; T007 suspect), '
         f'{sv["Trade_Date"].min():%Y-%m-%d} to {sv["Trade_Date"].max():%Y-%m-%d}. None verified against broker records.', fontsize=10.5, color=INK2)
fig.text(0.05, 0.915, f'USD, recorded P&L (fee treatment unknown). Passed-checks only ({n_clean} trades): net {money_precise(flag_free["net_pnl"])}, '
         f'profit factor {flag_free["profit_factor"]:.2f}, max drawdown {money_precise(-flag_free["max_drawdown"])}.', fontsize=10.5, color=INK2)
fig.text(0.05, 0.012, 'Caveats: small sample (39 trades). All 4 flagged trades are included in the headline figures. '
         'T039 absent from ID sequence; same-day order assumed by Trade_ID. Not evidence of future performance.', fontsize=9, color=INK2)

fig.savefig(FIG_DIR / 'trading_dashboard.png', dpi=300)     # high-resolution raster
fig.savefig(FIG_DIR / 'trading_dashboard.pdf')              # vector
plt.show()

## Stage 9 - Business insights and recommendations
Numbers below are computed from the analysis above, so they update if the data changes.

In [ ]:
# Stage 9 - Reviewed analytical report for this fixed synthetic journal
# The narrative was manually audited for statistical and financial claims.
# This notebook is intentionally calibrated to the supplied exercise CSV.
from IPython.display import Markdown, display

reviewed_path = REPORT_DIR / 'analytical_report_reviewed.md'
assert reviewed_path.is_file(), 'Missing reviewed report: keep the complete project folder intact'
flag_free = sensitivity.loc['Excluding all 4 flagged trades']
assert total_trades == 39, 'Report narrative must be reviewed when the journal changes'
assert np.isclose(net_profit, -215.46), 'Report narrative must be reviewed when P&L changes'
assert int(flag_free['trades']) == 35 and np.isclose(flag_free['net_pnl'], -194.46)
assert np.isclose(abs(max_drawdown), 427.88)
report = reviewed_path.read_text(encoding='utf-8')
(REPORT_DIR / 'analytical_report.md').write_text(report, encoding='utf-8')
print('Reviewed analytical report exported to', (REPORT_DIR / 'analytical_report.md').relative_to(ROOT))
display(Markdown(report))


## Stage 10 - Reproducibility checks

In [ ]:
# Independent re-computation from the exported cleaned CSV (proves the file reproduces the headline numbers)
chk = pd.read_csv(PROCESSED_DIR / 'trading_journal_cleaned.csv', parse_dates=['Trade_Date'])
chk = chk.sort_values(['Trade_Date', 'Trade_ID'])
p2 = chk['Realized_PnL_USD'].to_numpy()
assert len(chk) == 39 and chk['Trade_ID'].is_unique
assert chk[['Entry_Price', 'Exit_Price', 'Position_Size_Lots', 'Realized_PnL_USD']].notna().all().all()
assert np.isclose(p2.sum(), net_profit)
assert np.isclose(max_dd(p2), abs(max_drawdown))
assert np.isclose(p2[p2 > 0].sum() / abs(p2[p2 < 0].sum()), gross_profit / abs(gross_loss))
assert (chk['Quality_Status'] != 'Passed_Checks').sum() == 4
assert np.isfinite(chk[['Entry_Price', 'Exit_Price', 'Position_Size_Lots', 'Realized_PnL_USD']].to_numpy(float)).all()
assert (chk[['Entry_Price', 'Exit_Price', 'Position_Size_Lots']] > 0).all().all()
for f in ('trading_dashboard.png', 'trading_dashboard.pdf'):
    assert (FIG_DIR / f).stat().st_size > 10_000, f
for f in ('analytical_report.md', 'cleaning_log.csv', 'data_quality_report.csv'):
    assert (REPORT_DIR / f).exists(), f
print('All reproducibility checks passed.')
print('Outputs (relative to project root):')
for folder in (PROCESSED_DIR, FIG_DIR, REPORT_DIR):
    for f in sorted(folder.iterdir()):
        print('  ', f.relative_to(ROOT))